# Exporting Data

## Loading Libraries

In [ ]:
# Numerical Computing
import numpy as np

# Data Manipulation
import pandas as pd

# Data Visualization
import seaborn as sns
import matplotlib.pyplot as plt

# PyArrow
import pyarrow as pa

# ZipFile
import zipfile

# Structure Query Language (SQL)
import sqlite3
import sqlalchemy as sa

### Dirty Devil Data

In [ ]:
url = 'https://github.com/mattharrison/datasets/raw/master'\
      '/data/dirtydevil.txt'

df = pd.read_csv(url, skiprows=lambda num: num <34 or num == 35,
                 sep='\t')

In [ ]:
def to_denver_time(df_, time_col, tz_col):
    return (df_
       .assign(**{tz_col: df_[tz_col].replace('MDT', 'MST7MDT')})
       .groupby(tz_col)
       [time_col]
       .transform(lambda s: pd.to_datetime(s)
           .dt.tz_localize(s.name, ambiguous=True)
           .dt.tz_convert('America/Denver'))
    )

In [ ]:
def tweak_river(df_):
    return (df_
     .assign(datetime=to_denver_time(df_, 'datetime', 'tz_cd'))
     .rename(columns={'144166_00060': 'cfs',
                      '144167_00065': 'gage_height'})
     .set_index('datetime')
    )

In [ ]:
dd = tweak_river(df)
dd

In [ ]:
type(dd.index)

### Creating CSV Files

In [ ]:
dd.to_csv('/tmp/dd.csv') 

In [ ]:
print(dd.head(5).to_csv())

In [ ]:
dd2 = pd.read_csv('/tmp/dd.csv', index_col='datetime')

### Reading ZIP Files with CSV

In [ ]:
def extract_csv_from_zip(zip_file, csv_file):
    with zipfile.ZipFile(zip_file) as z:
        z.extract(csv_file)

In [ ]:
extract_csv_from_zip('/tmp/dd.zip', 'dd.csv')

pd.from_csv('dd.csv')

### Exporting Excel

In [ ]:
dd.to_excel('/tmp/dd.xlsx')

In [ ]:
(dd
    .reset_index()
    .assign(datetime=lambda df_: df_.datetime.dt.tz_convert(tz=None))
    .set_index('datetime')
    .to_excel('/tmp/dd.xlsx')
    )

In [ ]:
writer = pd.ExcelWriter('/tmp/dd2.xlsx')

dd2 = (dd
    .reset_index()
    .assign(datetime=lambda df_: df_.datetime.dt.tz_convert(tz=None))
    .set_index('datetime')
    )

(dd2
    .loc['2010':'2010-12-31']
    .to_excel(writer, sheet_name='2010')
    )

(dd2
    .loc['2011':'2011-12-31']
    .to_excel(writer, sheet_name='2011')
    )

writer.save()

### Parquet

In [ ]:
dd.to_parquet('/tmp/dd.parquet')

In [ ]:
dd2 = pd.read_parquet('/tmp/dd.parquet')

dd2.equals(dd)

In [ ]:
pd.testing.assert_frame_equal(dd2, dd)

In [ ]:
dd.agency_cd.dtype

In [ ]:
dd2.agency_cd.dtype

In [ ]:
type(dd.agency_cd.dtype), type(dd2.agency_cd.dtype)

In [ ]:
pd.testing.assert_series_equal(dd2.agency_cd, dd.agency_cd, check_dtype=False)

### Feather

In [ ]:
dd.to_feather('/tmp/dd.fea')

In [ ]:
(dd
    .reset_index()
    .to_feather('/tmp/dd.fea')
    )

In [ ]:
dd2 = pd.read_feather('/tmp/dd.fea')

pd.testing.assert_frame_equal(dd, dd2)

In [ ]:
pd.testing.assert_frame_equal(dd2, dd, check_dtype=False)

### SQL

In [ ]:
con = sqlite3.connect('dd.db')

In [ ]:
dd.to_sql('dd', con, if_exists='replace')

In [ ]:
eng = sa.create_engine('sqlite:///dd.db')

In [ ]:
sa_con = eng.connect()

In [ ]:
dd2 = pd.read_sql('dd', sa_con, index_col='datetime', dtype_backend='pyarrow')  

dd2.equals(dd) 

In [ ]:
pd.testing.assert_frame_equal(dd2, dd, check_dtype=False)

In [ ]:
print(dd2)

In [ ]:
pd.testing.assert_frame_equal(dd2
    .reset_index()
    .assign(datetime=lambda df_: pd.to_datetime(df_.datetime, utc=True)
        .dt.tz_localize('America/Denver'))
    .set_index('datetime'),
    dd, check_dtype=False
    )

### JSON

In [ ]:
obj = dd.to_dict()

In [ ]:
dd.head(2).to_dict()

In [ ]:
dd2 = pd.DataFrame.from_dict(obj)

In [ ]:
pd.testing.assert_frame_equal(dd2, dd)

In [ ]:
pd.testing.assert_frame_equal((dd2.rename_axis(index='datetime')), dd)

In [ ]:
pd.testing.assert_frame_equal((dd2
    .rename_axis(axis='datetime')
    .astype(dict(dd.dtypes))), dd)

In [ ]:
dd.reset_index().to_json('/tmp/dd.json.gz', orient='records', index=False, lines=True)

In [ ]:
dd2 = (pd.read_json('/tmp/dd/json.gz', orient='records', lines=True, dtype_backend='pyarrow', engine='pyarrow')
    .assign(datetime=lambda df_: pd.to_datetime(df_.datetime, utc=True).dt.tz_convert('America/Denver')))

In [ ]:
pd.testing.assert_frame_equal(dd2, dd.reset_index())

In [ ]:
dd3 = (dd2
    .assign(datetime=lambda df_: pd.to_datetime(df_.datetime. unit'ms')
        .dt.tz_localize(tz='UTC')
        .dt.tz_convert('America/Denver'))
    .set_index('datetime')
)

In [ ]:
print(dd3)

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=3d9749e8-6b93-4fd5-ac78-a2c3c3cdb9a7' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>